# 04 — Kalman : masse cachée (R2C2 / plant)

Deux états. C = [1, 0] : on mesure l'air seulement.
Kalman infère T_masse via R_am.

Ici le **plant** a un état vrai (S2 du catalogue). Le R2C2 d'identification
n'a pas alpha_s,mass — le filtre interne du MPC non plus.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from basic_mpc.control.internal import r2c2_internal_from_plant
from basic_mpc.identification.pem import P0_SCALE
from basic_mpc.models.kalman import run_kalman
from basic_mpc.models.plant import ThermalPlant, literature_plant_params, synthetic_weather
from basic_mpc.models.r2c2 import discretize as discretize_r2

plant_p = literature_plant_params()
n = int(48 * 3600 / plant_p.dt_seconds)
w = synthetic_weather(n, plant_p.dt_seconds, seed=3)
heating = np.where(((np.arange(n) * plant_p.dt_seconds / 3600) % 24 < 7), 0.0, 40.0)
plant = ThermalPlant(params=plant_p, x0=np.array([19.0, 18.0]), seed=1)
traj = plant.simulate(w["t_ext"].to_numpy(), w["S"].to_numpy(), heating)

internal = r2c2_internal_from_plant(plant_p)
ad, bd = discretize_r2(internal)
q = np.diag([internal.process_noise_std**2, internal.process_noise_std_mass**2])
r = np.array([[internal.sensor_noise_std**2]])
u = np.column_stack([traj["t_ext"], traj["S"], traj["P"]])
y = traj["y"].to_numpy()
res = run_kalman(y, u, ad, bd, np.array([[1.0, 0.0]]), q, r,
                 np.array([y[0], y[0]]), np.diag([P0_SCALE, 4 * P0_SCALE]))

In [ ]:
hours = np.arange(n) * plant_p.dt_seconds / 3600
fig, axes = plt.subplots(2, 1, figsize=(8, 5.4), sharex=True)
axes[0].plot(hours, traj["ta_true"], color="#2c2416", label="air vrai")
axes[0].plot(hours, y, color="#8a7e6e", lw=0.6, label="y")
axes[0].plot(hours, res.x_filt[:, 0], color="#3d6b6b", label="air Kalman")
axes[0].set_ylabel("°C")
axes[0].legend(frameon=False, ncol=3)
axes[0].set_title("Mesure = air seulement")
axes[1].plot(hours, traj["tm_true"], color="#2c2416", label="masse vraie")
axes[1].plot(hours, res.x_filt[:, 1], color="#8c4a32", label="masse Kalman")
axes[1].set_xlabel("heures")
axes[1].set_ylabel("°C")
axes[1].set_title("État jamais vu par le capteur")
axes[1].legend(frameon=False)
plt.show()
rmse_m = float(np.sqrt(np.mean((res.x_filt[:, 1] - traj["tm_true"]) ** 2)))
print(f"RMSE masse (Kalman vs plant) {rmse_m:.3f} °C")